# Model 3 results: how rising CO2 changes temperature across India

**Question:** for each 1° square of India (about 110 km) and each month, how much warmer or cooler than its own 1961–1990 normal was it, and how much of that is explained by CO2 versus natural factors?

**The project chain:** people and income → emissions (Model 1) → CO2 in the air (Model 2) → **India's temperature (Model 3)**.

**One prediction = one square in one month**, for example "square 26.5°N 73.5°E, May 2020: +0.8°C". Maps, timelines and the all-India line are built from these single predictions.

**How models were judged:**

| Test | Trained on | Tested on (never seen) |
|---|---|---|
| Exam 1 | 1951–1990 | 1991–2002 |
| Exam 2 | 1951–2000 | 2001–2012 |
| Exam 3 (final) | 1951–2010 | 2011–2024 |
| Hidden regions | 1951–2010, with whole 5×5° blocks removed | those blocks, 2011–2024 |

**Main score, skill:** % smaller error than "every month is normal" (anomaly 0), over every square and month.

**Champion rule, fixed before training:** the reference is `linear_co2`. A challenger needs average skill more than 5 points higher AND must win at least 2 of 3 exams. Baselines and `natural_only` can't be champion.

Run this notebook after `python3 src/models/train_m3_india_temp.py`. It reads the saved results and refits the models (trained up to 2010) for the extra checks in sections 6 and 7.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src" / "models"))
from train_m3_india_temp import M3Model, MODELS, INPUT_COLS, TARGET, per_square_line, decimal_year

REP = ROOT / "reports/models/m3"
READY = ROOT / "data/model_ready"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

res = pd.read_csv(REP / "comparison.csv")
summary = json.loads((REP / "summary.json").read_text())
CHAMP = summary["champion"]
print("Champion:", CHAMP)
print("Reason:", summary["reason"])

## 1. Data overview: the three model-ready tables

Each model has its own table, built by a DVC stage from the raw files, so anyone can rebuild it with `dvc repro <stage>`. The tables have different units: country-year, world-year and square-month. They link through the **year**: the yearly CO2 and emissions values in the Model 3 table are the same world values the chain passes along. Nothing in any table is filled or interpolated; rows with missing values were dropped.

In [ ]:
tables = {"Model 1: m1_emissions.csv (country x year)": READY / "m1_emissions.csv",
          "Model 2: m2_co2_growth.csv (world x year)": READY / "m2_co2_growth.csv",
          "Model 3: m3_india_temp.csv (square x month)": READY / "m3_india_temp.csv"}
rows = []
for name, path in tables.items():
    t = pd.read_csv(path, nrows=None if "m3" not in path.name else 5)
    n = len(pd.read_csv(path, usecols=[0])) if "m3" in path.name else len(t)
    rows.append({"table": name, "rows": n, "columns": len(t.columns),
                 "size_MB": round(path.stat().st_size / 1e6, 2),
                 "column names": ", ".join(t.columns)})
overview = pd.DataFrame(rows).set_index("table")
pd.set_option("display.max_colwidth", 400)
overview

In [ ]:
df = pd.read_csv(READY / "m3_india_temp.csv", usecols=INPUT_COLS + [TARGET, "co2_ppm"])
df = df.sort_values(["year", "month", "cell_id"]).reset_index(drop=True)
print(f"Model 3 table: {len(df):,} rows = {df.cell_id.nunique()} squares x months, "
      f"{df.year.min()}-{df.year.max()}")
print("Training rows (<=2010):", (df.year <= 2010).sum(), "| test rows (2011-2024):", (df.year > 2010).sum())
df.sample(5, random_state=0)

**How to read this:** each row is one square in one month. `tmean_anom` is the target. `co2_forcing` (CO2's warming push, in W/m², from CO2 ppm), `cum_emissions_gtco2` (total CO2 humans have emitted, world) and the `nino_*` columns (El Niño in earlier months) are the inputs. `lat`, `lon` and `month_sin`/`month_cos` say where and when. CO2 and emissions are yearly world values, so all squares share them in a given year.

## 2. Scores: every model in every exam

In [ ]:
order = list(MODELS)
skill = res.pivot(index="model", columns="test_years", values="skill_vs_zero_pct").loc[order]
skill["average"] = skill.mean(axis=1)
reg = pd.read_csv(REP / "region_holdout.csv", index_col=0)
skill["hidden regions"] = reg["hidden regions"].reindex(order)
final = res[res.exam == 2010].set_index("model").loc[order]
table = skill.round(1).join(final[["india_yearly_mae", "india_yearly_r2",
                                   "trend_predicted_per_decade"]].round(3))
table.index = [m + ("  ★ champion" if m == CHAMP else "") for m in table.index]
print("Skill in % (monthly, every square); India-yearly and trend columns are for 2011-2024")
print("Actual 2011-2024 trend:", round(final["trend_actual_per_decade"].iloc[0], 3), "degC per decade")
table

In [ ]:
ax = skill.drop(columns=["average", "hidden regions"]).plot.bar(figsize=(12, 5), width=0.8)
ax.axhline(0, color="k", lw=1)
ax.set_ylabel("% smaller error than 'every month is normal'")
ax.set_title("Skill in each exam (monthly, every square)")
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

**How to read this graph:** each group is one model, and each bar is one exam. Above zero means better than "every month is normal".

**What to notice:**
- Almost every model gets better from exam 1 to exam 3. With more training years, the models learn the CO2 effect better. Before about 1990, India barely warmed while CO2 rose (part of the warming was probably hidden by air pollution, a known explanation that our model can't test), so a model trained only up to 1990 learns a weak CO2 effect.
- The models with CO2 beat `natural_only` in exams 2 and 3.
- The hybrid with trees is the worst: extra complexity made it worse.

**Why the numbers look small:** a monthly value in one square swings by about ±1°C because of weather (heatwaves, clouds, late rain). The climate trend is a few tenths of a degree. Weather can't be predicted years ahead, so most monthly error is out of every model's reach. Sections 3 and 6 remove the weather by averaging.

## 3. All-India yearly: natural factors alone vs adding CO2

In [ ]:
yearly = pd.read_csv(REP / "india_yearly.csv", index_col=0)
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(yearly.index, yearly["actual"], "k-", lw=2.5, label="actual")
for m, c in [("natural_only", "tab:green"), ("linear_co2", "tab:red"),
             ("linear_cum_emissions", "tab:purple"), ("baseline_own_trend", "tab:gray")]:
    ax.plot(yearly.index, yearly[m], lw=2, color=c, label=m + (" (champion)" if m == CHAMP else ""))
ax.axvline(2010.5, color="grey", ls=":")
ax.text(2011, ax.get_ylim()[1] * 0.9, "unseen years →")
ax.axhline(0, color="k", lw=0.6)
ax.set_ylabel("all-India average anomaly (degC)")
ax.set_title("Models trained on 1951-2010: only the CO2/emissions models follow the warming")
ax.legend(loc="upper left"); plt.tight_layout(); plt.show()

hc = summary["human_cause_check"]["test_2011_2024_trend_per_decade"]
fig, ax = plt.subplots(figsize=(7, 3.5))
names = list(hc)
ax.barh(names, [hc[n] for n in names],
        color=["k" if n == "actual" else "tab:green" if n == "natural_only" else "tab:red" for n in names])
for i, n in enumerate(names):
    ax.text(hc[n] + 0.005, i, f"{hc[n]:+.3f}", va="center")
ax.set_xlabel("warming 2011-2024 (degC per decade)")
ax.set_title("Human-cause check on unseen years"); ax.invert_yaxis()
plt.tight_layout(); plt.show()

**How to read these graphs:** the black line is India's real yearly average anomaly, and the coloured lines are predictions from models trained only on 1951–2010. Right of the dotted line, every year is unseen. The bar chart compares the warming rate in those unseen years.

**What to notice:**
- **This is the project's main evidence.** The natural-only model (El Niño, place and season) stays almost flat and predicts only a small part of the 2011–2024 warming. Adding CO2 predicts most of it, using only what the model learned before 2011.
- The grey line, each square's own straight-line trend, also misses the recent speed-up. CO2 has risen faster and faster, and the CO2 model follows that acceleration while a straight line can't.
- The emissions-based model behaves almost the same as the CO2 model. Total emissions and CO2 in the air rise together (that is Model 2's link).

This shows a strong link that agrees with physics, not a laboratory proof. The model also slightly **underestimates** recent warming, most likely because it has no air-pollution input.

## 4. What the champion learned

In [ ]:
train = df[df.year <= 2010]
test = df[df.year > 2010]
fitted = {}
for name in MODELS:
    fitted[name] = M3Model(**MODELS[name]).fit(train[INPUT_COLS], train[TARGET])
    print("fitted", name)

champ = fitted[CHAMP]
if hasattr(champ, "linear_"):
    w = pd.Series(champ.linear_.coef_, index=champ.feature_names_)
    print("\nChampion weights (degC of anomaly per unit of each input):")
    print(w.round(4).to_string())
    col = MODELS[CHAMP].get("co2_col")
    if col == "co2_forcing":
        f = df.groupby("year")["co2_forcing"].first()
        print(f"\nCO2 forcing {f.loc[2010]:.2f} W/m2 (2010) -> {f.loc[2024]:.2f} (2024): "
              f"model attributes {w[col] * (f.loc[2024] - f.loc[2010]):+.2f} degC to that rise")

**How to read this:** each weight says how much the predicted anomaly changes when that input rises by 1. The CO2 weight is the key one: degrees of India warming per 1 W/m² of extra CO2 forcing. The last line turns it into the warming the model links to the CO2 rise from 2010 to 2024. El Niño weights are positive: warm El Niño months are followed by warmer months in India.

## 5. Works in places it has never seen?

In [ ]:
ax = reg.plot.bar(figsize=(10, 4.5))
ax.axhline(0, color="k", lw=1)
ax.set_ylabel("skill vs zero (%), 2011-2024")
ax.set_title("Normal test vs hidden regions")
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()
reg.round(1)

**How to read this graph:** for each model, the left bar is the normal final exam, where the model has seen every square's past. The right bar is the hidden-regions test, where whole 5×5° blocks (about 550 km wide) were removed from training, past included.

**What to notice:** the linear CO2 models score almost the same on hidden regions. They don't depend on memorising each square, so they should work for a district at the edge of the data. The hybrid drops, which fits it having learned patterns specific to the training data. `baseline_own_trend` and `linear_co2_per_square` can't take this test, because they need each square's own history.

## 6. Regional accuracy on unseen years (weather averaged out)

Section 2 scores single months, where weather dominates. Here, for each square, we average all 168 months of 2011–2024: actual vs predicted. Averaging cancels most of the weather and leaves the climate signal. This asks: **does the model get each region's recent warming right?**

Two scores:
- **Regional error (MAE):** the typical miss of a square's 2011–2024 average, in °C. Smaller is better.
- **Pattern match (correlation across squares):** does the model know *which* squares warmed more? 1 means a perfect ranking, 0 means no idea. The champion has one India-wide CO2 effect, but latitude and longitude are inputs, so it can still make a simple north–south or east–west tilt. Its score shows how much a tilt alone explains.

Note: this check was added after seeing the warming map, so it's a second measure. It doesn't change the champion rule.

In [ ]:
sq_actual = test.groupby("cell_id")[TARGET].mean()
rows, sq_pred = [], {}
for name, m in fitted.items():
    p = pd.Series(m.predict(test[INPUT_COLS]), index=test.index).groupby(test["cell_id"]).mean()
    sq_pred[name] = p
    err = p - sq_actual
    rows.append({"model": name,
                 "regional_MAE_degC": err.abs().mean(),
                 "pattern_correlation": sq_actual.corr(p) if p.std() > 1e-9 else np.nan,
                 "mean_bias_degC": err.mean()})
regional = pd.DataFrame(rows).set_index("model").round(3)
regional.index = [m + ("  ★ champion" if m == CHAMP else "") for m in regional.index]
print(f"Actual 2011-2024 average anomaly over squares: {sq_actual.mean():+.3f} degC "
      f"(squares range {sq_actual.min():+.2f} to {sq_actual.max():+.2f})")
regional

In [ ]:
cells = df.drop_duplicates("cell_id").set_index("cell_id")[["lat", "lon"]]
show = list(dict.fromkeys([CHAMP, "linear_co2_regional", "linear_co2_per_square"]))
panels = [("actual 2011-2024", sq_actual)] + [(m, sq_pred[m]) for m in show]
vmax = float(np.nanmax(np.abs(np.concatenate([p.values for _, p in panels]))))
fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 6), sharey=True)
for ax, (title, s) in zip(axes, panels):
    d = cells.join(s.rename("v"))
    sc = ax.scatter(d.lon, d.lat, c=d.v, cmap="RdBu_r", vmin=-vmax, vmax=vmax, marker="s", s=45)
    ax.set_title(title + (" (champion)" if title == CHAMP else ""), fontsize=10)
    ax.set_aspect("equal"); ax.set_xlabel("longitude")
axes[0].set_ylabel("latitude")
fig.colorbar(sc, ax=axes, shrink=0.7, label="average anomaly 2011-2024 (degC)")
fig.suptitle("Unseen years only: how warm was each square, and what did each model predict?")
plt.show()

fig, axes = plt.subplots(1, len(show), figsize=(5 * len(show), 4.5), sharey=True)
for ax, m in zip(axes, show):
    ax.scatter(sq_actual, sq_pred[m].reindex(sq_actual.index), s=12, alpha=0.6)
    lo, hi = sq_actual.min(), sq_actual.max()
    ax.plot([lo, hi], [lo, hi], "k--", lw=1)
    ax.set_title(m, fontsize=10); ax.set_xlabel("actual (degC)")
axes[0].set_ylabel("predicted (degC)")
fig.suptitle("Each dot = one square's 2011-2024 average (on the dashed line = perfect)")
plt.tight_layout(); plt.show()

**How to read these graphs:** the maps show each square's average anomaly over the unseen years 2011–2024, red meaning warmer than its 1961–1990 normal. In the scatter plots, each dot is one square: right means it really was warmer, up means the model predicted warmer. Dots on the dashed line are perfect. A flat cloud of dots means the model gives every square the same value.

**What to look for:**
- **Champion map:** at most a smooth tilt across India, because it has one India-wide CO2 effect plus latitude and longitude. It gets the overall level, not local details like the Bihar warming hole.
- **Regional and per-square:** do they put red and pale in the right places? Their pattern correlation in the table answers this.
- **The rule of thumb:** if a regional model has a regional MAE at least 10% smaller than the champion's and a higher pattern correlation, it's the better choice for the dashboard's district map. The champion stays the model for India-wide numbers and scenarios.

In [ ]:
r = regional.copy(); r.index = [i.replace("  ★ champion", "") for i in r.index]
cands = [m for m in ["linear_co2_regional", "linear_co2_per_square"] if m in r.index]
best = r.loc[cands, "regional_MAE_degC"].idxmin()
print(f"Champion ({CHAMP}): regional MAE {r.loc[CHAMP, 'regional_MAE_degC']:.3f} degC, "
      f"pattern correlation {r.loc[CHAMP, 'pattern_correlation']}")
print(f"Best regional model ({best}): regional MAE {r.loc[best, 'regional_MAE_degC']:.3f} degC, "
      f"pattern correlation {r.loc[best, 'pattern_correlation']:.2f}")
better = (r.loc[best, "regional_MAE_degC"] < 0.9 * r.loc[CHAMP, "regional_MAE_degC"]
          and r.loc[best, "pattern_correlation"] > r.loc[CHAMP, "pattern_correlation"])
print("\nSuggested use:", f"{best} for the district map, {CHAMP} for India-wide numbers and scenarios"
      if better else f"{CHAMP} for everything; no regional model clearly gets the pattern right")

## 7. The long-term warming map (with a caution)

In [ ]:
sq = pd.read_csv(REP / "square_trends.csv", index_col=0)
cols = [c for c in sq.columns if c not in ("lat", "lon")]
vmax = float(np.nanmax(np.abs(sq[cols].to_numpy())))
fig, axes = plt.subplots(1, len(cols), figsize=(5 * len(cols), 6), sharey=True)
for ax, c in zip(axes, cols):
    sc = ax.scatter(sq.lon, sq.lat, c=sq[c], cmap="RdBu_r", vmin=-vmax, vmax=vmax, marker="s", s=45)
    ax.set_title(c + (" (champion)" if c == CHAMP else ""), fontsize=10)
    ax.set_aspect("equal"); ax.set_xlabel("longitude")
axes[0].set_ylabel("latitude")
fig.colorbar(sc, ax=axes, shrink=0.7, label="warming 1951-2024 (degC per decade)")
plt.show()

**How to read this graph:** each square's warming rate over the whole period 1951–2024, from the data and from the models.

**What to notice:**
- **Real data:** fastest warming in west Rajasthan, Gujarat and the far north (Kashmir), plus the south. There's a near-zero "warming hole" over east Uttar Pradesh and Bihar, matching the EDA.
- **Per-square model:** reproduces this pattern closely.

**Caution:** 60 of these 74 years were training years, so part of the match is the model repeating history it learned. Section 6 is the fair, unseen-years version of this map.

## 8. Conclusions for the report

1. **CO2 explains India's recent warming; natural factors don't.** On unseen years (2011–2024), the natural-only model predicts only a small part of the real warming. Adding CO2 predicts most of it (see section 3 for the exact rates). This is our model's own human-cause evidence and completes the chain from emissions to CO2 to India's temperature.
2. **Champion: `linear_co2`**, by the rule fixed in advance. It beats "every month is normal" in exams 2 and 3, explains about half of the year-to-year changes in India's average temperature on unseen years, and works equally well in regions it never saw.
3. **Emissions work as well as CO2** (`linear_cum_emissions`, same average skill). The dashboard can use it for personal emissions what-ifs.
4. **Simpler is better.** Regional detail gave small gains only with many training years, and the tree hybrid was worst, as in Model 2.
5. **Regional detail:** section 6 shows whether a regional model gets each area's recent warming right. If it does, the dashboard map uses it, labelled as "regional detail, not the champion".
6. **Monthly predictions are modest by nature.** Weather dominates single months, so the useful outputs are yearly, multi-year and regional averages, and scenarios.

## 9. Honest limits

- **No air-pollution (aerosol) input.** Pollution probably hid part of India's warming before 1990, so the model learns a CO2 effect that is too weak at first and slightly underestimates recent warming. This is future work.
- **This is a statistical link that agrees with physics, not a controlled experiment.**
- **Future CO2 is beyond the training range,** so scenarios are what-ifs with a stated range, not forecasts. Linear models are used because they can extend beyond the training range, while trees can't (shown in Model 2).
- **Rows are not independent facts.** Neighbouring squares and months are alike. The final exam has 14 independent years, not 57,792 independent rows.
- **The IMD gridded data are estimates** built from station records, not thermometers in every square. Squares with more than 30% missing months (islands, far north-east, far north) were dropped.
- **The CO2 and emissions inputs are world values, not Indian ones.** The cumulative emissions cover fossil fuels and industry only.
- **The saved models were trained up to 2010.** For the dashboard (Phase 2), the champion will be retrained on all years up to 2024 and registered in MLflow.